In [1]:
%pip install torch torchvision opacus -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# Differential Privacy vs Non-DP SGD Comparison on MNIST
# ============================================================

import torch
from torch import nn, optim
from torchvision import datasets, transforms
from opacus import PrivacyEngine
import numpy as np

# -----------------------------
# CONFIGURATION
# -----------------------------
EPOCHS = 5
BATCH_SIZE = 128
LR = 0.1
MAX_GRAD_NORM = 1.0
TARGET_DELTA = 1e-5

# Dynamically generated epsilon values (modify as needed)
EPSILONS = list(range(1, 26, 4))   # Example: 1,5,9,13,17,21,25

# Storage
results = {
    "non_dp": {"acc": [], "loss": []},
    "dp": {}
}

# -----------------------------
# DATA
# -----------------------------
transform = transforms.Compose([transforms.ToTensor()])

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(".", train=True, download=True, transform=transform),
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    datasets.MNIST(".", train=False, download=True, transform=transform),
    batch_size=1024,
    shuffle=False
)

# -----------------------------
# EVALUATION FUNCTION
# -----------------------------
def evaluate(model):
    model.eval()
    correct = 0
    total = 0
    loss_fn = nn.CrossEntropyLoss()
    test_loss = 0

    with torch.no_grad():
        for x, y in test_loader:
            x = x.view(-1, 28 * 28)
            preds = model(x)
            test_loss += loss_fn(preds, y).item()
            correct += (preds.argmax(1) == y).sum().item()
            total += y.size(0)

    return correct / total, test_loss / len(test_loader)


# -----------------------------
# MODEL FACTORY
# -----------------------------
def create_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 128),
        nn.ReLU(),
        nn.Linear(128, 10)
    )

In [3]:
# -----------------------------
# 1. TRAIN NON-DP BASELINE
# -----------------------------
model = create_model()
optimizer = optim.SGD(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss()

print("\nTraining Non-DP SGD")

for epoch in range(EPOCHS):
    model.train()
    for x, y in train_loader:
        x = x.view(-1, 28 * 28)
        optimizer.zero_grad()
        preds = model(x)
        loss = loss_fn(preds, y)
        loss.backward()
        optimizer.step()

    acc, test_loss = evaluate(model)
    results["non_dp"]["acc"].append(acc)
    results["non_dp"]["loss"].append(test_loss)

    print(f"Epoch {epoch+1}: acc={acc:.4f} loss={test_loss:.4f}")



Training Non-DP SGD
Epoch 1: acc=0.9140 loss=0.3045
Epoch 2: acc=0.9300 loss=0.2425
Epoch 3: acc=0.9397 loss=0.2094
Epoch 4: acc=0.9466 loss=0.1825
Epoch 5: acc=0.9527 loss=0.1618


In [4]:
# -----------------------------
# 2. TRAIN DP-SGD MODELS
# -----------------------------
for epsilon in EPSILONS:

    print(f"\nTraining DP-SGD with target epsilon={epsilon}")

    model = create_model()
    optimizer = optim.SGD(model.parameters(), lr=LR)
    privacy_engine = PrivacyEngine()

    model, optimizer, dp_loader = privacy_engine.make_private_with_epsilon(
        module=model,
        optimizer=optimizer,
        data_loader=train_loader,
        target_epsilon=epsilon,
        target_delta=TARGET_DELTA,
        epochs=EPOCHS,
        max_grad_norm=MAX_GRAD_NORM
    )

    loss_fn = nn.CrossEntropyLoss()

    acc_list = []
    loss_list = []

    for epoch in range(EPOCHS):
        model.train()
        for x, y in dp_loader:
            x = x.view(-1, 28 * 28)
            optimizer.zero_grad()
            preds = model(x)
            loss = loss_fn(preds, y)
            loss.backward()
            optimizer.step()

        acc, test_loss = evaluate(model)
        acc_list.append(acc)
        loss_list.append(test_loss)

        print(f"Epoch {epoch+1}: acc={acc:.4f} loss={test_loss:.4f}")

    results["dp"][epsilon] = {
        "acc": acc_list,
        "loss": loss_list
    }



Training DP-SGD with target epsilon=1


c:\Users\Adi\AppData\Local\Programs\Python\Python311\Lib\site-packages\opacus\privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(
c:\Users\Adi\AppData\Local\Programs\Python\Python311\Lib\site-packages\opacus\accountants\analysis\rdp.py:332: UserWarning: Optimal order is the largest alpha. Please consider expanding the range of alphas to get a tighter privacy bound.
  warnings.warn(
C:\Users\Adi\AppData\Local\Temp\ipykernel_26228\1585792816.py:34: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


Epoch 1: acc=0.8020 loss=0.6448
Epoch 2: acc=0.8628 loss=0.4541
Epoch 3: acc=0.8807 loss=0.4140
Epoch 4: acc=0.8900 loss=0.4006
Epoch 5: acc=0.8947 loss=0.3924

Training DP-SGD with target epsilon=5
Epoch 1: acc=0.8062 loss=0.6363
Epoch 2: acc=0.8650 loss=0.4471
Epoch 3: acc=0.8826 loss=0.4075
Epoch 4: acc=0.8891 loss=0.3947
Epoch 5: acc=0.8953 loss=0.3902

Training DP-SGD with target epsilon=9
Epoch 1: acc=0.7895 loss=0.6581
Epoch 2: acc=0.8627 loss=0.4512
Epoch 3: acc=0.8812 loss=0.4070
Epoch 4: acc=0.8897 loss=0.3954
Epoch 5: acc=0.8942 loss=0.3921

Training DP-SGD with target epsilon=13
Epoch 1: acc=0.8050 loss=0.6237
Epoch 2: acc=0.8646 loss=0.4394
Epoch 3: acc=0.8830 loss=0.4018
Epoch 4: acc=0.8915 loss=0.3927
Epoch 5: acc=0.8961 loss=0.3892

Training DP-SGD with target epsilon=17
Epoch 1: acc=0.8016 loss=0.6473
Epoch 2: acc=0.8621 loss=0.4554
Epoch 3: acc=0.8796 loss=0.4116
Epoch 4: acc=0.8902 loss=0.3962
Epoch 5: acc=0.8950 loss=0.3942

Training DP-SGD with target epsilon=21
Ep

In [5]:
# ============================================================
# MODEL COMPARISON ENGINE
# ============================================================

def compare_models(results_dict):

    comparison = []

    # Non-DP baseline
    comparison.append({
        "model_type": "Non-DP",
        "epsilon": None,
        "final_acc": results_dict["non_dp"]["acc"][-1],
        "final_loss": results_dict["non_dp"]["loss"][-1]
    })

    # DP models (automatic)
    for eps, metrics in results_dict["dp"].items():
        comparison.append({
            "model_type": "DP",
            "epsilon": eps,
            "final_acc": metrics["acc"][-1],
            "final_loss": metrics["loss"][-1]
        })

    return comparison


def analyze_models(comparison, loss_threshold=None):

    print("\n==============================")
    print("FINAL MODEL COMPARISON")
    print("==============================")

    for m in comparison:
        print(m)

    # Maximum Accuracy
    best_acc = max(comparison, key=lambda x: x["final_acc"])
    print("\nModel with Maximum Accuracy:")
    print(best_acc)

    # Minimum Loss
    best_loss = min(comparison, key=lambda x: x["final_loss"])
    print("\nModel with Minimum Loss:")
    print(best_loss)

    # Optional constraint
    if loss_threshold is not None:
        filtered = [m for m in comparison if m["final_loss"] <= loss_threshold]

        if filtered:
            best_under_constraint = max(filtered, key=lambda x: x["final_acc"])
            print(f"\nBest Accuracy with Loss <= {loss_threshold}:")
            print(best_under_constraint)
        else:
            print(f"\nNo model satisfies loss <= {loss_threshold}")

    # Ranking
    ranked = sorted(comparison, key=lambda x: x["final_acc"], reverse=True)

    print("\nModels Ranked by Accuracy:")
    for r in ranked:
        print(r)

    return {
        "best_accuracy": best_acc,
        "best_loss": best_loss,
        "ranked": ranked
    }


# -----------------------------
# RUN COMPARISON
# -----------------------------
comparison_results = compare_models(results)

analysis = analyze_models(
    comparison_results,
    loss_threshold=0.30  # Set to None if no constraint needed
)


FINAL MODEL COMPARISON
{'model_type': 'Non-DP', 'epsilon': None, 'final_acc': 0.9527, 'final_loss': 0.16179151833057404}
{'model_type': 'DP', 'epsilon': 1, 'final_acc': 0.8947, 'final_loss': 0.39237558394670485}
{'model_type': 'DP', 'epsilon': 5, 'final_acc': 0.8953, 'final_loss': 0.3901773378252983}
{'model_type': 'DP', 'epsilon': 9, 'final_acc': 0.8942, 'final_loss': 0.3920665293931961}
{'model_type': 'DP', 'epsilon': 13, 'final_acc': 0.8961, 'final_loss': 0.38916355818510057}
{'model_type': 'DP', 'epsilon': 17, 'final_acc': 0.895, 'final_loss': 0.3942448765039444}
{'model_type': 'DP', 'epsilon': 21, 'final_acc': 0.8948, 'final_loss': 0.38824881613254547}
{'model_type': 'DP', 'epsilon': 25, 'final_acc': 0.8962, 'final_loss': 0.3945435225963593}

Model with Maximum Accuracy:
{'model_type': 'Non-DP', 'epsilon': None, 'final_acc': 0.9527, 'final_loss': 0.16179151833057404}

Model with Minimum Loss:
{'model_type': 'Non-DP', 'epsilon': None, 'final_acc': 0.9527, 'final_loss': 0.161791518